In [ ]:
# 全局设置
import datetime as dt
import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings('ignore', category=PerformanceWarning)
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False# 正确显示负号
from IPython.display import HTML

from QuantStudio import __QS_MainPath__

# 风险模型检验

风险模型检验用于评估风险模型对组合风险预测的准确度。QuantStudio 的风险模型检验功能位于 `BackTest.Risk` 子模块，采用 Bias Tests 方法，建立在回测基本框架之上。

> **前置阅读**：回测框架的整体架构、`BTNode`/`BTReport` 基类、执行流程（三层嵌套 `Cache → Context → Engine`）以及"算子-节点分离"设计模式，请先参阅 **[基本框架](基本框架.ipynb)**。Bias 统计量的数学定义与推导（Z-Score、滚动 Bias、稳健 Bias、RAD 统计量等）详见 **[风险模型](../风险模型/风险模型.ipynb#风险模型检验)**。

## 模块概览

`QuantStudio.BackTest.Risk` 提供以下测试组件：

| 组件 | 算子（计算层） | 回测节点（报告层） | 功能 |
|------|---------------|-------------------|------|
| Bias 检验 | `CalcPortfolioVolatility` / `CalcRandomPortfolio` | `BiasTest` | 检验风险模型对各类投资组合的风险预测准确度 |

该模块遵循"算子 + 回测节点"两层架构，其中 `BiasTest` 节点内部自动生成了多个投资组合并计算每个组合的 Z-Score 和 Bias 统计量。

## 理论基础概述

Bias 检验的核心思想是：将组合已实现收益率 $r_t$ 除以模型预测的风险 $\hat{\sigma}_t$ 得到 Z-Score，若模型预测准确，Z-Score 的标准差应接近 1。在此基础上计算滚动 Bias 统计量（Z-Score 的滚动标准差）和稳健 Bias 统计量（截尾版本），并与 95% 置信区间 $[1-\sqrt{2/T}, 1+\sqrt{2/T}]$ 比较，判断模型是低估还是高估了风险。此外，RAD 统计量（Bias 统计量偏离 1 的绝对均值）用于衡量平均预测偏差。

`BiasTest` 自动生成以下几类投资组合来全面检验风险模型：

1. **全体股票组合**：市值加权和等权的全部 A 股组合
2. **行业组合**：市值加权和等权的各行业组合
3. **行业中性组合**：各行业内因子值排名前一半/后一半的股票组成的等权组合
4. **风格因子组合**：风格因子的 Top 20%/Bottom 20% 组合（5 分位组合）
5. **随机组合**：市值加权和等权的随机抽取 N 只股票的组合（默认 N ∈ {20, 50, 100, 200}）

## BiasTest — 回测报告节点

`BiasTest` 是继承自 `BTNode` 的回测节点，内部自动生成多种类型的投资组合（全体组合、行业组合、行业中性组合、风格因子 Top/Bottom 组合、随机组合），计算每个组合的 Z-Score、Bias 统计量和稳健 Bias 统计量，并汇总成 HTML 报告。

### 构造参数

```python
BiasTest(
    descriptor_ids, price, risk_table, mask=None,
    weight_list=[DataFactor(1, args={"Name": "等权"})],
    style_list=[], industry=None,
    industry_neutral_factor_list=[], extra_portfolio_list=[],
    args={}, config_file=None, **kwargs
)
```

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `descriptor_ids` | `List[str]` | — | 依赖因子的截面 ID 序列 |
| `price` | `Factor` | — | 证券价格/净值因子，用于计算组合收益率 |
| `risk_table` | `RiskTable` | — | 风险表对象，提供协方差矩阵用于计算组合波动率 |
| `mask` | `Optional[Factor]` | `None` | 筛选条件因子，值为 1 的截面才参与计算 |
| `weight_list` | `List[Factor]` | `[等权因子]` | 权重因子列表，默认元素为等权因子 |
| `style_list` | `List[Factor]` | `[]` | 风格因子列表（用于生成风格 Top/Bottom 组合） |
| `industry` | `Optional[Factor]` | `None` | 行业因子，非 None 时自动生成行业组合和行业中性组合 |
| `industry_neutral_factor_list` | `List[Factor]` | `[]` | 行业中性因子列表（在各行业内按因子值取 Top/Bottom 50%） |
| `extra_portfolio_list` | `List[Factor]` | `[]` | 额外自定义投资组合因子列表 |
| `args` | `dict` | `{}` | 参数集 |

### args 参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"Bias Test"` | 节点名称（冻结） |
| `IndustryList` | `List[str]` | `[]` | 行业列表（冻结，指定 `industry` 时必填） |
| `RandomNums` | `List[int]` | `[20, 50, 100, 200]` | 随机组合的持仓数量列表（冻结） |
| `RebalanceDTs` | `Optional[List[datetime]]` | `None` | 再平衡时点，None 表示每个时点都再平衡（冻结） |
| `RollingAvgPeriod` | `int` | `12` | Bias 统计量滚动窗口期数（冻结） |
| `GenReport` | `bool` | `False` | 是否自动生成报告 |

### 自动生成的投资组合

`BiasTest` 在构造时对每类投资组合（每个权重因子 × 每个策略）自动生成一个因子：

**全体组合**（每个权重因子 1 个）：
- `"全体{权重}加权组合"`：所有股票的市值/等权组合

**行业组合**（每个权重因子 × 每个行业 1 个，需要指定 `industry` 和 `IndustryList`）：
- `"{行业}行业{权重}加权组合"`：单个行业内股票的市值/等权组合

**行业中性组合**（每个权重因子 × 每个行业中性因子 × Top/Bottom，需要指定 `industry` 和 `industry_neutral_factor_list`）：
- `"{因子}行业中性Top{权重}加权组合"`：各行业内因子值排名前 50% 的股票组合
- `"{因子}行业中性Bottom{权重}加权组合"`：各行业内因子值排名后 50% 的股票组合

**风格因子组合**（每个权重因子 × 每个风格因子 × Top/Bottom，需要指定 `style_list`）：
- `"{风格}风格Top{权重}加权组合"`：因子值排名前 20% 的股票组合
- `"{风格}风格Bottom{权重}加权组合"`：因子值排名后 20% 的股票组合

**随机组合**（每个权重因子 × 每个 `RandomNums` 元素 1 个）：
- `"随机{N}{权重}加权组合"`：随机选取 N 只股票的组合

> **注意**：如果指定了行业因子，`IndustryList` 参数不能为空，且参数中列出的行业名称必须与行业因子中实际存在的行业名称匹配。

### 构造示例

```python
from QuantStudio.BackTest.Risk.BiasTest import BiasTest
from QuantStudio.Factor.Factor import DataFactor

# 等权因子（默认就有的权重因子）
EqualWeight = DataFactor(1, args={"Name": "等权"})

BiasTestNode = BiasTest(
    descriptor_ids=SectionIDs,
    price=Price,
    risk_table=RT,
    mask=Mask,
    industry=Industry,
    args={
        "RebalanceDTs": BalanceDTs,
        "IndustryList": ["TMT", "Ind", "Fin"],
        "RandomNums": [5],
        "RollingAvgPeriod": 3,
        "GenReport": True
    }
)
```

### 输出数据结构

`BiasTest.backward_compute` 返回的字典包含：

| 键 | 类型 | 说明 |
|------|------|------|
| `"Z-Score"` | `DataFrame` | 各组合各期的 Z-Score 时序（$z_t = r_t / \hat{\sigma}_t$） |
| `"Robust Z-Score"` | `DataFrame` | 经截尾处理（$[-3, 3]$ 区间）的 Z-Score |
| `"Bias 统计量"` | `DataFrame` | 滚动 Bias 统计量时序，前两列为 95% 置信上下界 |
| `"Robust Bias 统计量"` | `DataFrame` | 滚动稳健 Bias 统计量时序，前两列为 95% 置信上下界 |
| `"汇总统计量"` | `DataFrame` | 各组合的 RAD、高估/低估比例、准确度汇总 |

**汇总统计量指标说明**：
- `RAD 统计量`：$|b_t^T - 1|$ 的均值，越接近 0 表示预测越准确
- `Robust RAD 统计量`：基于稳健 Bias 统计量的 RAD
- `Bias 统计量高估比例`：Bias 统计量低于 95% 置信下界的比例（高估风险）
- `Bias 统计量低估比例`：Bias 统计量高于 95% 置信上界的比例（低估风险）
- `Bias 统计量准确度`：Bias 统计量落在置信区间内的比例
- `Robust *`：对应的稳健版本

## 支持的辅助算子

`BiasTest` 内部使用了几个辅助算子，了解它们有助于深入理解 Bias 检验的计算流程。

### CalcPortfolioVolatility — 组合波动率计算算子

`CalcPortfolioVolatility` 是一个 `SectionOperator`，计算投资组合基于风险模型的波动率预测值 $\hat{\sigma}_t = \sqrt{w^T \mathbf{V} w}$，其中 $\mathbf{V}$ 是风险模型提供的协方差矩阵。

**`__call__` 参数**：

| 参数 | 类型 | 说明 |
|------|------|------|
| `*portfolio` | `Factor` | 投资组合因子列表（权重因子） |
| `risk_table` | `RiskTable` | 风险表对象 |
| `portfolio_name_list` | `Optional[List[str]]` | 组合名称列表 |

### CalcPortfolioReturn — 组合收益率计算算子

位于 `BackTest.Strategy.AllocationStrategy`，计算投资组合的已实现收益率 $r_t$。

### CalcRandomPortfolio — 随机组合生成算子

`CalcRandomPortfolio` 是一个 `SectionOperator`，生成随机选取 N 只股票的投资组合。

**`__call__` 参数**：

| 参数 | 类型 | 说明 |
|------|------|------|
| `weight` | `Factor` | 权重因子（如等权或市值加权） |
| `mask` | `Optional[Factor]` | 筛选条件因子 |

### Bias 检验的计算流程

`BiasTest` 节点在构造时内部执行的步骤：

1. **生成投资组合**：根据 `weight_list`、`industry`、`style_list`、`industry_neutral_factor_list`、`RandomNums` 等参数，通过 `_genPortfolio` 方法生成大量投资组合因子
2. **计算组合收益率**：通过 `CalcPortfolioReturn` 计算每个组合各期的已实现收益率 $r_t$
3. **计算组合波动率**：通过 `CalcPortfolioVolatility` 从风险表中读取协方差矩阵 $\mathbf{V}$，计算每个组合各期的预测波动率 $\hat{\sigma}_t = \sqrt{w^T \mathbf{V} w}$
4. **计算 Z-Score**：$z_t = r_t / \text{Lag}(\hat{\sigma}_t)$，其中波动率取滞后一期（上月预测 × 当月已实现）
5. **汇总分析**：在 `backward_compute` 中计算滚动 Bias 统计量、稳健 Bias 统计量、RAD 统计量等

## 完整示例：Bias 检验

下面展示一个完整的风险模型 Bias 检验流程。首先连接风险数据库，查看风险数据表的基本信息，然后构建 `BiasTest` 回测节点并执行。

> **执行流程**（三层嵌套 `Cache → Context → Engine`）以及 `BTReport` 的报告汇总机制已在 **[基本框架](基本框架.ipynb)** 中详细说明，此处不再赘述。

In [ ]:
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Core.Node import DTLocalContext, DTInitData
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Factor.FactorCache import FeatherFactorCache
from QuantStudio.Factor.HDF5DB import HDF5DB
from QuantStudio.Risk.HDF5RDB import HDF5FRDB
from QuantStudio.BackTest.BackTestModel import BTReport
from QuantStudio.Tools.DateTimeFun import getMonthLastDateTime

FDB = HDF5DB(args={"MainDir": Path(__QS_MainPath__).parent / "docs/data/HDF5"}).connect()
RDB = HDF5FRDB(args={"MainDir": Path(__QS_MainPath__).parent / "docs/data/Risk"}).connect()

# 查看风险数据表
RT = RDB.getTable("demo_risk_table")
print(f"风险表时点数: {len(RT.getDateTime())}")
RT.getDateTime()[-5:]

In [ ]:
from QuantStudio.BackTest.Risk.BiasTest import BiasTest

StartDT, EndDT = dt.datetime(2025, 1, 1), dt.datetime(2025, 4, 30)# 数据起止时间
TestStartDT, TestEndDT = dt.datetime(2025, 2, 28), EndDT# 测试起止时间

FT = FDB.getTable("stock_cn_day_bar")
DTRuler = FT.getDateTime(start_dt=StartDT, end_dt=EndDT)
TestDTs = FT.getDateTime(start_dt=TestStartDT, end_dt=TestEndDT)
SectionIDs = IDs = FT.getID()

DTs = FT.getDateTime(ifactor_name="close", start_dt=StartDT, end_dt=EndDT)

# 再平衡时点序列
BalanceDTs = getMonthLastDateTime(DTs)# 月末

# 获取数据因子
Price = FDB.getTable("stock_cn_day_bar").getFactor("close")
IfListed = FDB.getTable("stock_cn_status").getFactor("if_listed")
Industry = FDB.getTable("stock_cn_industry").getFactor("industry")
Mask = (IfListed == 1)

RT = RDB.getTable("demo_risk_table")

# 构建 Bias Test 回测节点
BiasTestNode = BiasTest(
    descriptor_ids=SectionIDs,
    price=Price,
    risk_table=RT,
    mask=Mask,
    industry=Industry,
    args={"RebalanceDTs": BalanceDTs, "RandomNums": [5], "IndustryList": ["TMT", "Ind", "Fin"], "RollingAvgPeriod": 3, "GenReport": True}
)

# 创建报告并执行
NodeList = [BiasTestNode]
Report = BTReport(bt_node_list=NodeList)

with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": "../data/Cache", "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Rslt = ExecEngine.run([Report], Context, fwd_data_list=[DTLocalContext(DTs=TestDTs)], init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))])

display(HTML(Rslt[0]["Report"]))

## 输出结构

报告 HTML 展示了各投资组合的 Bias 统计量和稳健 Bias 统计量，以及汇总统计表。`BiasTest` 的 `backward_compute` 返回字典的各键可通过编程方式访问各项统计量：

```python
# 访问 BiasTest 节点的输出
bias_output = Rslt[0]["0-Bias Test"]  # 键格式: "{序号}-{节点Name}"

# 查看 Z-Score 时序
z_score = bias_output["Z-Score"]           # DataFrame: index=时点, columns=组合名称

# 查看 Bias 统计量时序（含 95% 置信区间）
bias_stats = bias_output["Bias 统计量"]     # DataFrame: 前两列为置信区间，后续为各组合

# 查看汇总统计
summary = bias_output["汇总统计量"]          # DataFrame: index=组合名称, columns=各统计指标
rad = summary["RAD 统计量"]                  # 各组合的 RAD 统计量
accuracy = summary["Bias 统计量准确度"]      # 各组合的准确度
```

### 解读要点

1. **Bias 统计量**越接近 1 表示风险预测越准确
   - $b_t^T > 1 + \sqrt{2/T}$ 表示低估了实际风险
   - $b_t^T < 1 - \sqrt{2/T}$ 表示高估了实际风险
2. **稳健 Bias 统计量**排除了异常收益率的影响，应重点关注
3. **RAD 统计量**反映平均偏差水平，越小越好（理想值约 0.17）
4. **准确度**反映 Bias 统计量落在置信区间内的比例，越高越好
5. 不同类型的投资组合（行业组合、风格组合、随机组合）对模型的不同方面进行检验，应综合判断